# Project 2 — Environment and Cloud setup

Run this notebook on the Mac's Jupyter kernel for local preflight and Cloud setup. Spark analysis and model training run only on the Dataproc Jupyter kernel.

The run is pinned to project `sg-rainfall-dsa5208-261011`, region `asia-southeast1`, Education billing account `01DAD8-AE5F1A-E3910A`, and the user-authorised credit ID `3LU63BPGY77JFDQQ`. Google applies credits automatically at billing-account scope; `gcloud` cannot attach a credit ID to an individual job. This workflow checks project/account linkage and records the authorised credit identity. It does not require Reports or Cost-table review.

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess

from IPython.display import display

PROJECT_ID = "sg-rainfall-dsa5208-261011"
BILLING_ACCOUNT_NAME = "Billing Account for Education"
BILLING_ACCOUNT_ID = "01DAD8-AE5F1A-E3910A"
REGION = "asia-southeast1"
CLUSTER = "dsa5208-rainfall-fixed8"
BUCKET = "sg-rainfall-dsa5208-261011-data"
RUN_ID = os.environ.get("RUN_ID", "20261016-v2-fixed8-04")
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"

In [ ]:
DATA_ROOT_VALUE = os.environ.get("DATA_ROOT")
if not DATA_ROOT_VALUE:
    raise RuntimeError("Set DATA_ROOT to the directory containing the raw/ input folder.")
DATA_ROOT = Path(DATA_ROOT_VALUE).expanduser()
if not (DATA_ROOT / "raw").is_dir():
    raise FileNotFoundError(DATA_ROOT / "raw")


def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "scripts").is_dir() and (candidate / "manifests").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from this notebook.")


REPO_ROOT = find_repo_root()
print(f"Notebook root: {REPO_ROOT}")
print(f"Project: {PROJECT_ID}")
print(f"Billing account: {BILLING_ACCOUNT_NAME} ({BILLING_ACCOUNT_ID})")
print(f"Run ID: {RUN_ID}")


In [ ]:
REBUILD_MANIFEST = False  # Set True to rescan the eight local annual inputs.
manifest_path = REPO_ROOT / "manifests" / "input_manifest.json"
if REBUILD_MANIFEST:
    import importlib.util

    module_path = REPO_ROOT / "scripts" / "build_manifest.py"
    spec = importlib.util.spec_from_file_location("build_manifest", module_path)
    manifest_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(manifest_module)
    rebuilt = manifest_module.build_manifest(DATA_ROOT)
    temporary_path = manifest_path.with_suffix(".json.tmp")
    temporary_path.write_text(json.dumps(rebuilt, indent=2) + "\n", encoding="utf-8")
    temporary_path.replace(manifest_path)
    print(f"Rebuilt manifest: {manifest_path}")


In [ ]:
if not manifest_path.is_file():
    raise FileNotFoundError(manifest_path)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_rows = [
    {
        "year": item["year"],
        "rows": item["row_count"],
        "size_bytes": item["size_bytes"],
        "sha256": item["sha256"],
        "schema_matches": item["schema_matches_expected"],
        "malformed_rows": item["rows_with_wrong_field_count"],
        "invalid_timestamps": item["unparseable_or_missing_timestamps"],
    }
    for item in manifest["files"]
]
display(manifest_rows)
print(f"Input years: {manifest['selection']['years']}")
print(f"Files in manifest: {len(manifest_rows)}")
print("Raw CSV files are read-only.")


In [ ]:
CREATE_PROJECT = True

if CREATE_PROJECT:
    result = subprocess.run(
        ["gcloud", "projects", "describe", PROJECT_ID, "--format=json"],
        capture_output=True, text=True,
    )
    if result.returncode:
        created = subprocess.run(
            ["gcloud", "projects", "create", PROJECT_ID, "--name=DSA5208 Rainfall"],
            capture_output=True, text=True,
        )
        if created.returncode:
            raise RuntimeError("Project creation failed. Check project-creation permissions in Console.")
        print(f"Created project: {PROJECT_ID}")
    else:
        print(f"Project already exists: {PROJECT_ID}")

In [ ]:
import re

def run_gcloud(args, *, json_output=False):
    result = subprocess.run(
        ["gcloud", *args, *([f"--billing-project={PROJECT_ID}"] if args[:2] == ["billing", "budgets"] else [])],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode:
        message = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+", "[redacted identity]", result.stderr.strip())
        raise RuntimeError(message or f"gcloud exited with {result.returncode}")
    if json_output:
        return json.loads(result.stdout or "null")
    if result.stdout.strip():
        print(result.stdout.strip())
    return result.stdout


In [ ]:
def billing_project_state():
    return run_gcloud(
        ["billing", "projects", "describe", PROJECT_ID, "--format=json"],
        json_output=True,
    )

before = billing_project_state()
target = f"billingAccounts/{BILLING_ACCOUNT_ID}"
print({
    "project_id": PROJECT_ID,
    "billing_enabled_before": before.get("billingEnabled"),
    "already_linked_to_education": before.get("billingAccountName") == target,
})


In [ ]:
LINK_PROJECT_TO_EDUCATION = True

def ensure_education_billing_link():
    current = billing_project_state()
    if current.get("billingAccountName") == target and current.get("billingEnabled") is True:
        return current, True

    try:
        run_gcloud([
            "billing", "projects", "link", PROJECT_ID,
            f"--billing-account={BILLING_ACCOUNT_ID}",
        ])
        verified = billing_project_state()
        billing_ready = (
            verified.get("billingAccountName") == target
            and verified.get("billingEnabled") is True
        )
    except RuntimeError as exc:
        message = str(exc)
        if "IAM_PERMISSION_DENIED" in message or "does not have permission" in message:
            print(
                "Billing link blocked: the active gcloud identity lacks "
                "project billing-assignment permissions. No bucket, cluster, or job was created."
            )
            return current, False
        raise
    return verified, billing_ready


In [ ]:
if LINK_PROJECT_TO_EDUCATION:
    verified, billing_ready = ensure_education_billing_link()
    display({
        "project_id": PROJECT_ID,
        "billing_account": BILLING_ACCOUNT_NAME,
        "billing_account_id": BILLING_ACCOUNT_ID,
        "billing_enabled": verified.get("billingEnabled"),
        "linkage_verified": verified.get("billingAccountName") == target,
    })
else:
    billing_ready = False
    print("Skipped billing linkage. No cloud resources may be created.")


In [ ]:
# Stop before provisioning if the project is not linked to the selected account.
if not billing_ready:
    raise RuntimeError(
        "Project billing is not enabled on the selected Education account. "
        "No cloud resources will be created."
    )
print("The selected project billing account is verified for this setup session.")


In [ ]:
ENABLE_APIS = True
REQUIRED_APIS = [
    "cloudresourcemanager.googleapis.com",
    "serviceusage.googleapis.com",
    "iap.googleapis.com",
    "dataproc.googleapis.com",
    "compute.googleapis.com",
    "storage.googleapis.com",
    "iam.googleapis.com",
    "cloudbilling.googleapis.com",
]

if ENABLE_APIS:
    for api in REQUIRED_APIS:
        run_gcloud(["services", "enable", api, f"--project={PROJECT_ID}"])
else:
    print("API enablement skipped.")


In [ ]:
# Read live quotas and the measured fixed Dataproc profile.
region_info = run_gcloud(
    ["compute", "regions", "describe", REGION, f"--project={PROJECT_ID}", "--format=json"],
    json_output=True,
)
project_info = run_gcloud(
    ["compute", "project-info", "describe", f"--project={PROJECT_ID}", "--format=json"],
    json_output=True,
)
experiment_config = json.loads(
    (REPO_ROOT / "config" / "experiment.json").read_text(encoding="utf-8")
)
compute_profile = experiment_config["compute_profile"]

display({
    "regional_quotas": [
        quota for quota in region_info.get("quotas", [])
        if quota["metric"] in ("CPUS", "E2_CPUS", "DISKS_TOTAL_GB", "SSD_TOTAL_GB", "IN_USE_ADDRESSES")
    ],
    "project_quotas": [
        quota for quota in project_info.get("quotas", [])
        if quota["metric"] in ("CPUS_ALL_REGIONS", "DISKS_TOTAL_GB", "SSD_TOTAL_GB")
    ],
})
display({
    "region": REGION,
    "compute_profile": compute_profile,
    "disk_sizing_evidence": compute_profile["disk_sizing_basis"],
    "spark_event_log_uri": f"gs://{BUCKET}/spark-history/",
    "lifecycle": compute_profile["lifecycle"],
})


In [ ]:
display({
    "project_id": PROJECT_ID,
    "billing_account": BILLING_ACCOUNT_NAME,
    "billing_account_id": BILLING_ACCOUNT_ID,
    "billing_linkage_verified": billing_ready,
})
print("Notebook setup uses this single selected account; science notebooks do not repeat the check.")


In [ ]:
NETWORK = "rainfall-private"
SUBNET_NAME = "rainfall-singapore"
VM_SERVICE_ACCOUNT = f"rainfall-vm@{PROJECT_ID}.iam.gserviceaccount.com"

def resource_exists(args):
    return subprocess.run(["gcloud", *args, "--format=json"], capture_output=True, text=True).returncode == 0

if not resource_exists(["compute", "networks", "describe", NETWORK, f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "networks", "create", NETWORK, "--subnet-mode=custom", f"--project={PROJECT_ID}"])
if not resource_exists(["compute", "networks", "subnets", "describe", SUBNET_NAME, f"--region={REGION}", f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "networks", "subnets", "create", SUBNET_NAME, f"--network={NETWORK}", f"--region={REGION}", "--range=10.20.0.0/24", "--enable-private-ip-google-access", f"--project={PROJECT_ID}"])
if not resource_exists(["compute", "firewall-rules", "describe", "rainfall-internal", f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "firewall-rules", "create", "rainfall-internal", f"--network={NETWORK}", "--source-ranges=10.20.0.0/24", "--allow=tcp,udp,icmp", f"--project={PROJECT_ID}"])
if not resource_exists(["iam", "service-accounts", "describe", VM_SERVICE_ACCOUNT, f"--project={PROJECT_ID}"]):
    run_gcloud(["iam", "service-accounts", "create", "rainfall-vm", "--display-name=Rainfall Dataproc VM", f"--project={PROJECT_ID}"])
# Dedicated VM identity; no service-account keys are created.
run_gcloud(["projects", "add-iam-policy-binding", PROJECT_ID, f"--member=serviceAccount:{VM_SERVICE_ACCOUNT}", "--role=roles/dataproc.worker", "--condition=None", "--format=none"])
print("Private subnet, internal-only firewall and dedicated VM identity prepared.")
if not resource_exists(["compute", "firewall-rules", "describe", "rainfall-iap-ssh", f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "firewall-rules", "create", "rainfall-iap-ssh", f"--network={NETWORK}", "--source-ranges=35.235.240.0/20", "--allow=tcp:22", "--target-tags=rainfall-cluster", f"--project={PROJECT_ID}"])
# Reuse the signed-in project owner for IAP; do not expose the human identity.
active_identity = subprocess.run(["gcloud", "config", "get-value", "account"], capture_output=True, text=True, check=True).stdout.strip()
run_gcloud(["projects", "add-iam-policy-binding", PROJECT_ID, f"--member=user:{active_identity}", "--role=roles/iap.tunnelResourceAccessor", "--condition=None", "--format=none"])
print("IAP SSH access prepared for the existing project owner; no public VM address.")


In [ ]:
CREATE_BUCKET = True
UPLOAD_INPUTS = True

if CREATE_BUCKET and not resource_exists(["storage", "buckets", "describe", f"gs://{BUCKET}"]):
    run_gcloud([
        "storage", "buckets", "create", f"gs://{BUCKET}",
        f"--project={PROJECT_ID}",
        f"--location={REGION}",
        "--default-storage-class=STANDARD",
        "--uniform-bucket-level-access",
    ])
    print(f"Created bucket gs://{BUCKET}")

run_gcloud([
    "storage", "buckets", "add-iam-policy-binding", f"gs://{BUCKET}",
    f"--member=serviceAccount:{VM_SERVICE_ACCOUNT}",
    "--role=roles/storage.objectAdmin", "--format=none",
])


In [ ]:
if UPLOAD_INPUTS:
    manifest_path = REPO_ROOT / "manifests" / "input_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(manifest_path)
    manifest_uri = f"gs://{BUCKET}/runs/{RUN_ID}/config/input_manifest.json"
    if resource_exists(["storage", "objects", "describe", manifest_uri]):
        raise FileExistsError(f"Refusing to replace run manifest: {manifest_uri}")
    run_gcloud(["storage", "cp", str(manifest_path), manifest_uri])


In [ ]:
if UPLOAD_INPUTS:
    for record in manifest["files"]:
        year = record["year"]
        target_uri = f"gs://{BUCKET}/{record['relative_path']}"
        if resource_exists(["storage", "objects", "describe", target_uri]):
            object_info = run_gcloud(
                ["storage", "objects", "describe", target_uri, "--format=json"],
                json_output=True,
            )
            metadata = object_info.get("custom_fields", object_info.get("metadata", {}))
            if metadata.get("sha256") != record["sha256"]:
                raise RuntimeError(
                    f"Existing input has no matching SHA-256 metadata: {target_uri}"
                )
            print(f"Verified existing input: {target_uri}")
        else:
            source = DATA_ROOT / "raw" / f"HistoricalRainfallacrossSingapore{year}.csv"
            if not source.is_file():
                raise FileNotFoundError(source)
            run_gcloud([
                "storage", "cp", str(source), target_uri,
                f"--custom-metadata=sha256={record['sha256']}",
            ])
            print(f"Uploaded {year}: {record['size_bytes']} bytes")


In [ ]:
if UPLOAD_INPUTS:
    config_path = REPO_ROOT / "config" / "experiment.json"
    config_uri = f"gs://{BUCKET}/runs/{RUN_ID}/config/experiment.json"
    run_gcloud(["storage", "cp", str(config_path), config_uri])
    print(f"Inputs and experiment configuration uploaded for {RUN_ID}.")
else:
    print("Input upload skipped by configuration.")


In [ ]:
# Upload the analysis notebooks and shared Spark modeling helpers for this run.
import hashlib
import tempfile

analysis_notebooks = [
    "01_clean_and_task1.ipynb",
    "02_build_features.ipynb",
    "03_model_development.ipynb",
    "04_final_fit_evaluate.ipynb",
]
experiment_config = json.loads(
    (REPO_ROOT / "config" / "experiment.json").read_text(encoding="utf-8")
)
feature_source_run_id = experiment_config["base_features_source_run_id"]
if RUN_ID == feature_source_run_id:
    raise RuntimeError("Use a new run ID; the feature source run is immutable.")


def source_sha256(path):
    if path.suffix == ".ipynb":
        notebook = json.loads(path.read_text(encoding="utf-8"))
        source_only = {
            "cells": [
                {
                    "cell_type": cell["cell_type"],
                    "source": (
                        "".join(cell.get("source", []))
                        if isinstance(cell.get("source", ""), list)
                        else cell.get("source", "")
                    ),
                }
                for cell in notebook["cells"]
            ]
        }
        content = json.dumps(
            source_only, ensure_ascii=False, sort_keys=True, separators=(",", ":")
        ).encode("utf-8")
    else:
        content = path.read_bytes()
    return hashlib.sha256(content).hexdigest()


source_hashes = {
    "config/experiment.json": hashlib.sha256(
        (REPO_ROOT / "config" / "experiment.json").read_bytes()
    ).hexdigest(),
    "manifests/input_manifest.json": hashlib.sha256(
        (REPO_ROOT / "manifests" / "input_manifest.json").read_bytes()
    ).hexdigest(),
}
for name in analysis_notebooks:
    source_path = REPO_ROOT / "notebooks" / name
    run_gcloud([
        "storage", "cp", str(source_path),
        f"{ROOT}/source/notebooks/{name}",
    ])
    source_hashes[f"notebooks/{name}"] = source_sha256(source_path)

modeling_source = REPO_ROOT / "scripts" / "rainfall_modeling.py"
modeling_source_uri = f"{ROOT}/source/rainfall_modeling.py"
run_gcloud(["storage", "cp", str(modeling_source), modeling_source_uri])
source_hashes["scripts/rainfall_modeling.py"] = hashlib.sha256(
    modeling_source.read_bytes()
).hexdigest()

# Reuse the frozen feature artifact from the completed preprocessing run.
feature_source_uri = f"gs://{BUCKET}/runs/{feature_source_run_id}/base_features"
feature_target_uri = f"{ROOT}/base_features"
if not resource_exists(["storage", "objects", "describe", f"{feature_source_uri}/_SUCCESS"]):
    raise FileNotFoundError(f"Feature source is incomplete: {feature_source_uri}")
if resource_exists(["storage", "objects", "describe", f"{feature_target_uri}/_SUCCESS"]):
    print(f"Verified existing feature artifact: {feature_target_uri}")
else:
    run_gcloud([
        "storage", "rsync", "--recursive",
        feature_source_uri, feature_target_uri,
    ])
    if not resource_exists(["storage", "objects", "describe", f"{feature_target_uri}/_SUCCESS"]):
        raise RuntimeError(f"Feature copy did not complete: {feature_target_uri}")

source_manifest_uri = f"{ROOT}/config/source_hashes.json"
if resource_exists(["storage", "objects", "describe", source_manifest_uri]):
    raise FileExistsError(f"Refusing to replace source manifest: {source_manifest_uri}")
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", encoding="utf-8", delete=False) as handle:
    json.dump(source_hashes, handle, indent=2, sort_keys=True)
    source_manifest_path = Path(handle.name)
try:
    run_gcloud(["storage", "cp", str(source_manifest_path), source_manifest_uri])
finally:
    source_manifest_path.unlink(missing_ok=True)

print({
    "uploaded_notebooks": analysis_notebooks,
    "uploaded_source": modeling_source_uri,
    "feature_data_source_run_id": feature_source_run_id,
    "feature_data": feature_target_uri,
    "source_manifest": source_manifest_uri,
})

In [ ]:
CREATE_CLUSTER = True
experiment_config = json.loads(
    (REPO_ROOT / "config" / "experiment.json").read_text(encoding="utf-8")
)
profile = experiment_config["compute_profile"]
DATAPROC_IMAGE = profile["dataproc_image_version"]
SUBNET = SUBNET_NAME

cluster_exists = resource_exists([
    "dataproc", "clusters", "describe", CLUSTER,
    f"--project={PROJECT_ID}", f"--region={REGION}",
])
if CREATE_CLUSTER and not cluster_exists:
    if not BUCKET or not SUBNET or not VM_SERVICE_ACCOUNT or not DATAPROC_IMAGE:
        raise RuntimeError("Set bucket, subnet, service account, and exact image first.")
    master = profile["master"]
    workers = profile["workers"]
    spark = profile["spark"]
    lifecycle = profile["lifecycle"]
    properties = ",".join([
        f"dataproc:jupyter.notebook.gcs.dir=gs://{BUCKET}/notebooks/",
        "spark:spark.eventLog.enabled=true",
        f"spark:spark.ui.showConsoleProgress={str(spark['show_console_progress']).lower()}",
        f"spark:spark.eventLog.dir=gs://{BUCKET}/spark-history/",
        f"spark:spark.dynamicAllocation.enabled={str(spark['dynamic_allocation_enabled']).lower()}",
        f"spark:spark.executor.instances={spark['executor_instances']}",
        f"spark:spark.executor.cores={spark['executor_cores']}",
        f"spark:spark.sql.shuffle.partitions={experiment_config['shuffle_partitions']}",
    ])
    run_gcloud([
        "dataproc", "clusters", "create", CLUSTER,
        f"--project={PROJECT_ID}", f"--region={REGION}",
        f"--image-version={DATAPROC_IMAGE}",
        f"--subnet={SUBNET}", "--no-address",
        f"--service-account={VM_SERVICE_ACCOUNT}", "--scopes=cloud-platform",
        f"--master-machine-type={master['machine_type']}",
        f"--master-boot-disk-type={master['boot_disk_type']}",
        f"--master-boot-disk-size={master['boot_disk_size_gb']}GB",
        f"--num-workers={workers['count']}",
        f"--worker-machine-type={workers['machine_type']}",
        f"--worker-boot-disk-type={workers['boot_disk_type']}",
        f"--worker-boot-disk-size={workers['boot_disk_size_gb']}GB",
        f"--bucket={BUCKET}", f"--temp-bucket={BUCKET}",
        "--tags=rainfall-cluster", "--enable-component-gateway", "--optional-components=JUPYTER",
        f"--properties={properties}",
        f"--delete-max-idle={lifecycle['delete_max_idle']}",
        f"--delete-max-age={lifecycle['delete_max_age']}",
    ])
    print(f"Dataproc cluster {CLUSTER} is ready in {REGION}.")
elif cluster_exists:
    print("Cluster exists; run the profile check below before uploading or launching notebook jobs.")
else:
    print("Cluster creation was skipped.")

In [ ]:
experiment_config = json.loads(
    (REPO_ROOT / "config" / "experiment.json").read_text(encoding="utf-8")
)
profile = experiment_config["compute_profile"]
cluster_info = run_gcloud([
    "dataproc", "clusters", "describe", CLUSTER,
    f"--project={PROJECT_ID}", f"--region={REGION}", "--format=json",
], json_output=True)

cluster_config = cluster_info["config"]
actual_master = cluster_config["masterConfig"]
actual_workers = cluster_config["workerConfig"]
actual_properties = cluster_config.get("softwareConfig", {}).get("properties", {})


def short_machine_type(value):
    return str(value).rsplit("/", 1)[-1]


expected = {
    "master_machine_type": profile["master"]["machine_type"],
    "master_disk_type": profile["master"]["boot_disk_type"],
    "master_disk_gb": profile["master"]["boot_disk_size_gb"],
    "worker_count": profile["workers"]["count"],
    "worker_machine_type": profile["workers"]["machine_type"],
    "worker_disk_type": profile["workers"]["boot_disk_type"],
    "worker_disk_gb": profile["workers"]["boot_disk_size_gb"],
    "executor_instances": profile["spark"]["executor_instances"],
    "executor_cores": profile["spark"]["executor_cores"],
    "dynamic_allocation_enabled": str(profile["spark"]["dynamic_allocation_enabled"]).lower(),
    "show_console_progress": str(profile["spark"]["show_console_progress"]).lower(),
    "shuffle_partitions": experiment_config["shuffle_partitions"],
}
actual = {
    "master_machine_type": short_machine_type(actual_master["machineTypeUri"]),
    "master_disk_type": actual_master["diskConfig"]["bootDiskType"],
    "master_disk_gb": actual_master["diskConfig"]["bootDiskSizeGb"],
    "worker_count": actual_workers["numInstances"],
    "worker_machine_type": short_machine_type(actual_workers["machineTypeUri"]),
    "worker_disk_type": actual_workers["diskConfig"]["bootDiskType"],
    "worker_disk_gb": actual_workers["diskConfig"]["bootDiskSizeGb"],
    "executor_instances": int(actual_properties.get("spark:spark.executor.instances", "-1")),
    "executor_cores": int(actual_properties.get("spark:spark.executor.cores", "-1")),
    "dynamic_allocation_enabled": actual_properties.get(
        "spark:spark.dynamicAllocation.enabled", "missing"
    ).lower(),
    "show_console_progress": actual_properties.get(
        "spark:spark.ui.showConsoleProgress", "missing"
    ).lower(),
    "shuffle_partitions": int(
        actual_properties.get("spark:spark.sql.shuffle.partitions", "-1")
    ),
}
profile_matches = actual == expected

display({
    "cluster": CLUSTER,
    "state": cluster_info["status"]["state"],
    "expected_profile": expected,
    "actual_profile": actual,
    "profile_matches": profile_matches,
    "component_urls": cluster_config.get("endpointConfig", {}).get("httpPorts", {}),
    "lifecycle": cluster_config.get("lifecycleConfig"),
})
if not profile_matches:
    raise RuntimeError(
        "Cluster does not match the frozen compute profile; do not launch final-fit notebooks before the profile check passes."
    )

zone = cluster_config["gceClusterConfig"]["zoneUri"].rsplit("/", 1)[-1]
instance_names = actual_master.get("instanceNames", []) + actual_workers.get(
    "instanceNames", []
)
minimum_free_bytes = int(
    profile["disk_sizing_basis"]["target_free_headroom_gb"] * 1_000_000_000
)
disk_headroom = []
for instance_name in instance_names:
    filesystem_line = run_gcloud([
        "compute", "ssh", instance_name,
        f"--project={PROJECT_ID}", f"--zone={zone}",
        "--tunnel-through-iap",
        "--command=df -P -B1 / | tail -1",
    ]).strip()
    fields = filesystem_line.split()
    if len(fields) < 5:
        raise RuntimeError(f"Could not parse root filesystem usage for {instance_name}.")
    available_bytes = int(fields[3])
    disk_headroom.append({
        "instance": instance_name,
        "filesystem_bytes": int(fields[1]),
        "used_bytes": int(fields[2]),
        "free_bytes": available_bytes,
        "free_gb": round(available_bytes / 1_000_000_000, 2),
    })
display({"disk_headroom": disk_headroom})
if any(item["free_bytes"] < minimum_free_bytes for item in disk_headroom):
    raise RuntimeError(
        f"At least one node has less than "
        f"{profile['disk_sizing_basis']['target_free_headroom_gb']} GB free."
    )


In [ ]:
DELETE_CLUSTER = False

if DELETE_CLUSTER:
    run_gcloud([
        "dataproc", "clusters", "delete", CLUSTER,
        f"--project={PROJECT_ID}", f"--region={REGION}", "--quiet",
    ])
    remaining = run_gcloud([
        "dataproc", "clusters", "list",
        f"--project={PROJECT_ID}", f"--region={REGION}",
        "--format=json",
    ], json_output=True)
    display({"remaining_cluster_names": [item["clusterName"] for item in remaining]})
else:
    print("Cluster deletion skipped. Set DELETE_CLUSTER=True after outputs and notebooks are persisted.")

## Analysis notebooks

Run `01_clean_and_task1.ipynb`, `02_build_features.ipynb`,
`03_model_development.ipynb`, then `04_final_fit_evaluate.ipynb` on the
Dataproc Jupyter kernel. Notebook 03 reads only the 2017–2022 development
partitions and writes a frozen specification. Notebook 04 verifies that
specification before fitting the final model; it fits isotonic calibration on
2023, then reads the held-out 2024 partition only for final evaluation.